# FishCheck — EfficientNetB0 분류 모델 학습

**입력 데이터**: YOLO bbox 크롭 이미지 (광어 / 가자미, 499장)  
**모델**: EfficientNetB0 (ImageNet 사전학습 → Fine-tuning)  
**환경**: Google Colab T4 GPU

## 실행 순서
1. 런타임 → 런타임 유형 변경 → **T4 GPU** 선택
2. Section 2에서 크롭 이미지 업로드 (로컬 → Drive → Colab)
3. 셀 순서대로 전체 실행 (Ctrl+F9)
4. Section 6에서 effnet.pt HF Hub 업로드

## 1. 환경 설정 및 GPU 확인

In [ ]:
!pip install -q timm huggingface-hub

import torch
print(f'PyTorch : {torch.__version__}')
print(f'CUDA    : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU     : {torch.cuda.get_device_name(0)}')
else:
    print('⚠️  GPU 없음 — 런타임 유형을 T4 GPU 로 변경하세요')

## 2. 크롭 이미지 업로드

로컬에서 생성한 `data/effnet_crops/` 를 Google Drive에 올리거나 직접 압축해서 업로드합니다.

**방법 A (Google Drive 사용)**
1. `data/effnet_crops/` 폴더를 Drive의 `FishCheck/effnet_crops/` 에 업로드
2. 아래 Drive 마운트 셀 실행

**방법 B (직접 zip 업로드)**
- 로컬에서 `data/effnet_crops/` 를 zip 압축 → Colab 파일 탭에서 업로드

In [ ]:
# 방법 A: Google Drive 마운트
from google.colab import drive
drive.mount('/content/drive')

import shutil
from pathlib import Path

DRIVE_CROPS = Path('/content/drive/MyDrive/FishCheck/effnet_crops')
LOCAL_CROPS = Path('/content/effnet_crops')

if DRIVE_CROPS.exists():
    shutil.copytree(str(DRIVE_CROPS), str(LOCAL_CROPS))
    print(f'Drive에서 복사 완료: {LOCAL_CROPS}')
else:
    print(f'Drive 경로 없음: {DRIVE_CROPS}')
    print('방법 B로 직접 업로드하거나 Drive 경로를 확인하세요')

In [ ]:
# 데이터 확인
from pathlib import Path

CROPS_DIR = Path('/content/effnet_crops')
CLASSES   = sorted([d.name for d in CROPS_DIR.iterdir() if d.is_dir()])

print(f'클래스: {CLASSES}')
for cls in CLASSES:
    count = len(list((CROPS_DIR / cls).glob('*.jpg')))
    print(f'  {cls}: {count}장')

## 3. 데이터로더 구성

In [ ]:
import torch
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms
from pathlib import Path

CROPS_DIR  = Path('/content/effnet_crops')
BATCH_SIZE = 32
VAL_RATIO  = 0.2
SEED       = 42

train_tf = transforms.Compose([
    transforms.Resize((240, 240)),
    transforms.RandomCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

val_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

full_dataset = datasets.ImageFolder(CROPS_DIR)
CLASSES      = full_dataset.classes
n_val        = int(len(full_dataset) * VAL_RATIO)
n_train      = len(full_dataset) - n_val

torch.manual_seed(SEED)
train_ds, val_ds = random_split(full_dataset, [n_train, n_val])
train_ds.dataset.transform = train_tf
val_ds.dataset              = datasets.ImageFolder(CROPS_DIR, transform=val_tf)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print(f'클래스: {CLASSES}')
print(f'학습: {n_train}장  |  검증: {n_val}장')

## 4. EfficientNetB0 Fine-tuning

In [ ]:
import torch
import torch.nn as nn
import torchvision.models as models

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
EPOCHS = 30
LR     = 1e-3

# ImageNet 사전학습 가중치 로드 → 마지막 레이어만 교체
model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1)
model.classifier[1] = nn.Linear(model.classifier[1].in_features, len(CLASSES))
model = model.to(DEVICE)

# 초반 10 epoch: 분류 헤드만 학습 → 이후 전체 미세조정
for param in model.features.parameters():
    param.requires_grad = False

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.classifier.parameters(), lr=LR)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

best_acc, best_state = 0.0, None

for epoch in range(1, EPOCHS + 1):
    # 10 epoch 이후 전체 레이어 학습
    if epoch == 11:
        for param in model.features.parameters():
            param.requires_grad = True
        optimizer = torch.optim.Adam(model.parameters(), lr=LR * 0.1)
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS - 10)

    # 학습
    model.train()
    train_loss = 0.0
    for imgs, labels in train_loader:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        loss = criterion(model(imgs), labels)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
    scheduler.step()

    # 검증
    model.eval()
    correct = total = 0
    with torch.no_grad():
        for imgs, labels in val_loader:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            preds = model(imgs).argmax(dim=1)
            correct += (preds == labels).sum().item()
            total   += labels.size(0)

    acc = correct / total
    if acc > best_acc:
        best_acc   = acc
        best_state = {k: v.clone() for k, v in model.state_dict().items()}

    print(f'Epoch {epoch:02d}/{EPOCHS}  loss={train_loss/len(train_loader):.4f}  val_acc={acc*100:.1f}%  best={best_acc*100:.1f}%')

print(f'\n학습 완료 — best val_acc: {best_acc*100:.1f}%')

## 5. 가중치 저장 및 Drive 백업

In [ ]:
from pathlib import Path
import shutil

# best 가중치 복원 후 저장
model.load_state_dict(best_state)
torch.save(best_state, '/content/effnet.pt')
print('effnet.pt 저장 완료')

# Google Drive 백업
drive_out = Path('/content/drive/MyDrive/FishCheck/models')
drive_out.mkdir(parents=True, exist_ok=True)
shutil.copy('/content/effnet.pt', drive_out / 'effnet.pt')
print(f'Drive 백업 완료: {drive_out / "effnet.pt"}')

## 6. Hugging Face Hub 업로드

huggingface.co → Settings → Access Tokens → write 권한 토큰 준비

In [ ]:
from huggingface_hub import HfApi, login
from google.colab import userdata

HF_USERNAME = 'your-hf-username'  # ← 본인 계정으로 수정

try:
    login(token=userdata.get('HF_TOKEN'))
except Exception:
    login()  # 수동 입력

REPO_ID = f'{HF_USERNAME}/fishcheck-model'
api = HfApi()
api.create_repo(repo_id=REPO_ID, exist_ok=True)
api.upload_file(
    path_or_fileobj = '/content/effnet.pt',
    path_in_repo    = 'effnet.pt',
    repo_id         = REPO_ID,
)
print(f'업로드 완료 → {REPO_ID}/effnet.pt')